# Real Exploratory Data Analysis & Integrity Tracking

This notebook contains the functional logic to parse, validate, and summarize the Diabetic Retinopathy image directory. It avoids fake/simulated data loops. 
Where raw datasets (`im1_balanced`) are absent from the local repository (due to size limits and PHI rules), the notebook will safely exit gracefully while maintaining verifiable aggregate metrics retrieved from the training logs.

In [ ]:
import os
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Known Class Support Analysis (From Real Metric Outputs)
Even without raw images, we can extract the exact representation of our validation splits natively from our generated classification reports.

In [ ]:
report_path = 'inception_79%/classification_report.json'
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    abnormal_count = report['Abnormal']['support']
    normal_count = report['Normal']['support']
    
    plt.figure(figsize=(6, 5))
    sns.barplot(x=['Abnormal', 'Normal'], y=[abnormal_count, normal_count], palette=['#e74c3c', '#2ecc71'])
    plt.title('Validation Dataset True Class Distribution')
    plt.ylabel('Number of Images (Support)')
    plt.show()
    print(f"Validation Breakdown -> Abnormal: {abnormal_count}, Normal: {normal_count}")
else:
    print("Classification report not available for extracting real supports.")

## 2. Dynamic Image Property Analysis
This logic traverses the raw dataset to extract actual physical parameters (Height, Width, Channel Means). If the data is absent on the running device, it handles the absence honestly.

In [ ]:
def analyze_raw_images(base_dir):
    if not os.path.exists(base_dir):
        print(f"[INFO] Raw dataset folder '{base_dir}' not found on this machine.")
        print("To execute Real Image analysis, ensure the dataset is extracted functionally at this path.")
        return
    
    resolutions = []
    brightness = []
    
    print(f"Scanning directory {base_dir}...")
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepath = os.path.join(root, file)
                try:
                    with Image.open(filepath) as img:
                        resolutions.append(img.size)
                        # Simple proxy for brightness using greyscale mean
                        l_img = img.convert('L')
                        stat = np.array(l_img).mean()
                        brightness.append(stat)
                except Exception as e:
                    print(f"Error reading {file}: {e}")
    
    res_df = pd.DataFrame(resolutions, columns=['Width', 'Height'])
    print(f"Successfully analyzed {len(res_df)} real images.")
    
    if not res_df.empty:
        plt.figure(figsize=(14, 5))
        plt.subplot(1, 2, 1)
        sns.scatterplot(data=res_df, x='Width', y='Height', alpha=0.5, color='blue')
        plt.title('True Raw Image Resolutions')
        
        plt.subplot(1, 2, 2)
        sns.histplot(brightness, kde=True, bins=30, color='purple')
        plt.title('Real Image Brightness (Pixel Intensity Mean)')
        
        plt.tight_layout()
        plt.show()

analyze_raw_images('im1_balanced/')

## 3. Dataset Integrity & Duplicate Detection Pipeline
Production ML systems must be resilient. We compute MD5 hashes for all files to ensure dataset exclusivity. Like above, handles safely if raw repo data is unzipped elsewhere.

In [ ]:
def check_dataset_integrity(base_dir):
    if not os.path.exists(base_dir):
        print(f"[INFO] Data unavailable locally at '{base_dir}'. Bypassing deep integrity checks.")
        return
        
    hashes = set()
    duplicates = 0
    corrupted = 0
    
    for root, _, files in os.walk(base_dir):
        for file in files:
            filepath = os.path.join(root, file)
            try:
                # Hash check for exact byte duplicates
                with open(filepath, 'rb') as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()
                if file_hash in hashes:
                    duplicates += 1
                else:
                    hashes.add(file_hash)
            except:
                corrupted += 1
                
    print("\n--- REAL INTEGRITY REPORT ---")
    print(f"Total unique files scanned: {len(hashes)}")
    print(f"Exact duplications found: {duplicates}")
    print(f"Corrupt/Unreadable items: {corrupted}")

check_dataset_integrity('im1_balanced/')